# Hard-Negative Retraining of NRMS (Two-Stage)

This notebook trains an **NRMS** reranker with **hard negatives** mined by the Dense/MiniLM retriever, then benchmarks it through the retrieval stage. This is the most industry-aligned variant of the two-stage pipeline (retrieval -> reranking): instead of training on MIND's *random* negatives, we train the ranker on news that a content retriever actually surfaces for the user (retrieved-but-unclicked), which matches the production distribution the reranker will face.

## Pipeline
1. **Mine** — `DenseRetriever` (MiniLM) embeds the news corpus, then for each training impression retrieves the top-K news for the user's *history*; the retrieved-but-unshown items become **hard negatives**.
2. **Train** — `listwise_hn` mode builds listwise impressions `(history, [positives + mined hard negatives], labels)` and trains NRMS with the canonical listwise (masked softmax cross-entropy) loss.
3. **Benchmark** — the trained model is scored through the retrieval stage (TF-IDF / Dense / Entity) exactly like `02_retrieval_benchmark.ipynb`.

## Scale
- **Smoke test (this notebook, default):** small corpus slice + few impressions to verify the pipeline runs end-to-end without OOM.
- **Full run (big system):** set `MINE_MAX_NEWS = None`, `MAX_TRAIN_IMPRESSIONS = None`, `MAX_IMPRESSIONS = None`, and use a GPU.

In [ ]:
import os
import sys
import json
import argparse

import pandas as pd
import torch

sys.path.insert(0, '.')

from src.data import prepare_data
from src.model import build_default_nrms
from src.train import get_device
from src.common import prepare_run
from src.train_run import run_train
from src.retrieval_eval import run_benchmark, results_to_dataframe

## 1. Run Configuration

Set the smoke-test scale here. For the full run on a big system, set `MINE_MAX_NEWS = None`, `MAX_TRAIN_IMPRESSIONS = None`, `MAX_IMPRESSIONS = None`, and run on a GPU.

In [ ]:
DATA_TRAIN = 'data/MINDsmall_train'
DATA_DEV = 'data/MINDsmall_dev'
ENTITY_VEC = os.path.join(DATA_TRAIN, 'entity_embedding.vec')

# --- Smoke-test scale (small). Set to None for the full run on a big system. ---
MINE_MAX_NEWS = 2000       # cap on the hard-negative mining corpus (None = all ~65k news)
MAX_TRAIN_IMPRESSIONS = 200
MAX_DEV_IMPRESSIONS = 100
MAX_IMPRESSIONS = 30       # dev impressions evaluated in the benchmark
EPOCHS = 1
MINE_NUM_HN = 4            # hard negatives mined per impression (NRMS/MIND K)
K = 50                     # retrieved candidate set size (matches --max_candidates)
EVAL_BATCH_SIZE = 8

# Toggle retrievers in the benchmark
USE_TFIDF = True
USE_DENSE = True
USE_ENTITY = True

## 2. Build args + mine + prepare data

`prepare_run` calls `prepare_data` with `train_mode='listwise_hn'`, which mines hard negatives via the Dense/MiniLM retriever (capped to `MINE_MAX_NEWS` for the smoke test) and builds the listwise training set.

In [ ]:
args = argparse.Namespace(
    train_behaviors=os.path.join(DATA_TRAIN, 'behaviors.tsv'),
    train_news=os.path.join(DATA_TRAIN, 'news.tsv'),
    dev_behaviors=os.path.join(DATA_DEV, 'behaviors.tsv'),
    dev_news=os.path.join(DATA_DEV, 'news.tsv'),
    max_history_len=30,
    max_title_len=20,
    min_word_freq=2,
    max_train_impressions=MAX_TRAIN_IMPRESSIONS,
    max_dev_impressions=MAX_DEV_IMPRESSIONS,
    neg_samples=None,
    in_time_val_frac=0.0,
    in_time_val_seed=42,
    train_mode='listwise_hn',
    max_candidates=K,
    mine_num_hn=MINE_NUM_HN,
    mine_model='sentence-transformers/all-MiniLM-L6-v2',
    mine_cache_dir='cache',
    mine_max_news=MINE_MAX_NEWS,
    embed_dim=50,
    num_heads=5,
    user_num_heads=5,
    use_hf_embeddings=False,
    freeze_embeddings=False,
    hf_pool='mean',
    hf_cache='cache',
    bottleneck_dim=None,
    dropout=0.2,
    category_mode='none',
    cat_embed_dim=8,
    subcat_embed_dim=8,
    epochs=EPOCHS,
    steps_per_epoch=None,
    batch_size=32,
    eval_batch_size=EVAL_BATCH_SIZE,
    lr=5e-4,
    grad_clip=1.0,
    use_amp=False,
    num_workers=0,
    pos_weight=None,
    checkpoint_dir='checkpoints_hn',
    run_name='hn',
    save_every=1,
    early_stopping_patience=3,
    early_stopping_min_delta=0.0,
    seed=42,
    attribution=False,
    attribution_splits='dev',
    phase='all',
)

device = get_device()
state = prepare_run(args)
print(f"Mined training impressions: {len(state.train_loader.dataset)}")

## 3. Train (hard-negative NRMS)

One epoch on CPU for the smoke test. The loss is the canonical NRMS listwise (masked softmax cross-entropy) over `[positives + mined hard negatives]`.

In [ ]:
run_train(state)
print('Training complete.')
model = state.model
criterion = state.criterion

## 4. Benchmark (retrieval -> NRMS rerank)

`run_benchmark` builds the corpus index once, then evaluates each enabled retriever end-to-end with the hard-negative-trained model.

In [ ]:
results = run_benchmark(
    train_news_path=os.path.join(DATA_TRAIN, 'news.tsv'),
    dev_news_path=os.path.join(DATA_DEV, 'news.tsv'),
    dev_behaviors_path=os.path.join(DATA_DEV, 'behaviors.tsv'),
    entity_vec_path=ENTITY_VEC,
    model=model,
    device=device,
    criterion=criterion,
    k=K,
    max_impressions=MAX_IMPRESSIONS,
    eval_batch_size=EVAL_BATCH_SIZE,
    use_tfidf=USE_TFIDF,
    use_dense=USE_DENSE,
    use_entity=USE_ENTITY,
    max_news=MINE_MAX_NEWS,
)

## 5. Results

Side-by-side comparison of the three retrievers with the hard-negative-trained NRMS.

- `recall@k` / `hit_rate` — standalone retrieval quality (fraction of clicked news found in top-K).
- `retrieved_*` — end-to-end metrics after NRMS reranks the retrieved set.

> Note: on a smoke-test corpus slice the clicked positives usually live outside the small index, so `recall@k` is ~0 by design. The full run recovers meaningful recall. Compare `retrieved_mrr` / `retrieved_ndcg@5` against the baseline `runs_1784137259` from `02_retrieval_benchmark.ipynb`.

In [ ]:
df = results_to_dataframe(results)
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)
print(f"=== HARD-NEGATIVE (max_news={MINE_MAX_NEWS}, max_impressions={MAX_IMPRESSIONS}) ===")
df

## 6. Full run on a big system

For the full-scale run, change the config in cell 1:

```python
MINE_MAX_NEWS = None        # mine over the full ~65k corpus
MAX_TRAIN_IMPRESSIONS = None
MAX_IMPRESSIONS = None
EVAL_BATCH_SIZE = 64        # raise on GPU
EPOCHS = 5                  # or more
```

Then re-run cells 2–5. The Dense retriever batches corpus embedding internally, so it will not OOM on the full corpus given enough RAM / a GPU. On Modal, launch via `run_nrms_mind.py` with `train_mode='listwise_hn'` (it forwards `--mine_num_hn`, `--mine_model`, `--mine_cache_dir`, and `--mine_max_news`).